# EDA — `mouvements`

---

## Objectif de ce notebook

Ce notebook réalise l'analyse exploratoire de la table `mouvements`.  
Elle enregistre **toutes les opérations physiques** de déplacement de produits :  
livraisons, ventes, transferts, ajustements, pertes et retours fournisseurs.  

Son analyse révèle les patterns opérationnels de l'entreprise et guidera  
le paramétrage du modèle de détection d'anomalies (notebook 07).

---

## 0. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

# Dossier racine pour les figures
FIGURES_DIR = os.path.join('..', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Imports OK ✅')

## 1. Chargement des Données

In [ ]:
df = pd.read_csv('../data/mouvements.csv')
df['date'] = pd.to_datetime(df['date'])
df['annee'] = df['date'].dt.year
df['mois']  = df['date'].dt.month
df['heure_int'] = df['heure'].str.split(':').str[0].astype(int)
df['jour_semaine'] = df['date'].dt.dayofweek
df['quantite_abs'] = df['quantite'].abs()

print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période    : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Dépôts     : {df['depot_id'].nunique()}")
print(f"Produits   : {df['produit_id'].nunique()}")
print(f"Camions    : {df['camion_id'].nunique()}")
print(f"Opérateurs : {df['operateur_id'].nunique()}")
df.head(5)

## 2. Inspection Initiale

On examine les types de colonnes, les valeurs manquantes et les statistiques descriptives.

In [ ]:
print('=== Types de colonnes ===')
print(df.dtypes)
print()
print('=== Valeurs manquantes ===')
missing = df.isnull().sum()
print(missing[missing > 0])
print()
print(f'=== Doublons : {df.duplicated().sum()} ===')

In [ ]:
# Statistiques descriptives des variables numériques
df[['quantite', 'quantite_abs', 'valeur_mouvement', 'prix_unitaire']].describe().round(2)

In [ ]:
# Répartition des 6 types de mouvements
type_counts = df['type_mouvement'].value_counts()
print('=== Répartition des types de mouvements ===')
for t, n in type_counts.items():
    print(f'  {t:<30} : {n:>6,} ({n/len(df)*100:.1f}%)')

print()
print(f'=== Plage horaire des opérations ===')
print(f'  Heure min : {df["heure"].min()}')
print(f'  Heure max : {df["heure"].max()}')
print(f'  Opérations nocturnes (22h-6h) : {((df["heure_int"] >= 22) | (df["heure_int"] < 6)).sum():,}')

**📝 Observations :**

> Le dataset contient **42 529 mouvements** sans valeur manquante sur les colonnes essentielles. Deux colonnes ont des valeurs nulles légitimes : `camion_id` (8 562 nulls, soit 20,1% — normal pour les ajustements inventaire et pertes qui ne mobilisent pas de camion) et `bon_commande_ref` (29 705 nulls, soit 69,8% — seules les entrées livraison et retours fournisseurs ont une référence de bon de commande).
>
> La **Sortie Vente** domine avec 39,5% des mouvements (16 796 opérations), suivie de l'**Entrée Livraison** (30,2%). Les opérations couvrent 24h sur 24 — les opérations nocturnes (22h–6h) représentent une part à analyser pour détecter des comportements atypiques.
>
> La quantité est **signée** : positive pour les entrées, négative pour les sorties — ce qui est la convention comptable standard. La `quantite_abs` a été créée pour les analyses de volume.

## 3. Analyse par Type de Mouvement

On visualise la répartition, les volumes et les valeurs financières par type de mouvement.

In [ ]:
# Camembert et barplot de la répartition des types
try:
    type_counts = df['type_mouvement'].value_counts()
    colors = ['#14325A', '#1E8E5A', '#C9841A', '#C63B3B', '#5B6B7C', '#B8952F']

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    axes[0].pie(
        type_counts.values,
        labels=type_counts.index,
        autopct='%1.1f%%',
        colors=colors,
        startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2)
    )
    axes[0].set_title('Répartition des types de mouvements (nombre)')

    vol_type = df.groupby('type_mouvement')['quantite_abs'].sum().sort_values(ascending=True)
    vol_type.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
    axes[1].set_title('Volume total par type de mouvement (quantité absolue)')
    axes[1].set_xlabel('Volume total')
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, '01_repartition_types_mouvements.png'), dpi=150)
    plt.show()
except Exception as e:
    print('Erreur de visualisation :', e)
    plt.close('all')

In [ ]:
# Valeur financière totale par type
try:
    valeur_type = df.groupby('type_mouvement')['valeur_mouvement'].sum().sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 5))
    valeur_type.plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
    ax.set_title('Valeur financière totale par type de mouvement (USD)')
    ax.set_xlabel('Valeur totale (USD)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, '02_valeur_par_type.png'), dpi=150)
    plt.show()

    print('Valeur moyenne par mouvement :')
    print(df.groupby('type_mouvement')['valeur_mouvement'].mean().sort_values(ascending=False).apply(lambda x: f'{x:,.0f} USD'))
except Exception as e:
    print('Erreur de visualisation :', e)
    plt.close('all')

In [ ]:
# Évolution mensuelle des entrées et sorties sur 10 ans
entrees = df[df['type_mouvement'] == 'Entrée Livraison'].groupby(
    df['date'].dt.to_period('M'))['quantite_abs'].sum()
sorties = df[df['type_mouvement'] == 'Sortie Vente'].groupby(
    df['date'].dt.to_period('M'))['quantite_abs'].sum()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(entrees.index.astype(str), entrees.values,
        color='steelblue', linewidth=0.8, label='Entrées Livraison')
ax.plot(sorties.index.astype(str), sorties.values,
        color='crimson', linewidth=0.8, label='Sorties Vente')
ax.set_title('Évolution mensuelle des entrées et sorties (2015–2024)')
ax.set_xlabel('Mois')
ax.set_ylabel('Volume')
ax.set_xticks(range(0, len(entrees), 12))
ax.set_xticklabels([str(entrees.index[i]) for i in range(0, len(entrees), 12)], rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '03_evolution_entrees_sorties.png'), dpi=150)
plt.show()

**📝 Observations :**

> Les **Sorties Vente** représentent le plus grand volume financier — c'est le flux commercial principal de l'entreprise. Les **Entrées Livraison** sont moins fréquentes en nombre mais plus volumineuses unitairement (97,9M d'unités en volume total).
>
> Les **Pertes/Évaporation** (-16,2M d'unités) et les **Ajustements Inventaire** (-33M d'unités) sont des signaux importants — les ajustements négatifs importants peuvent indiquer des erreurs de gestion ou des détournements potentiels.
>
> L'évolution mensuelle montre une certaine régularité des flux entrées/sorties, avec des pics ponctuels probablement liés à des livraisons massives. Les entrées et sorties semblent bien corrélées dans le temps — ce qui est logique (on livre pour répondre aux ventes).

## 4. Analyse Temporelle et Opérationnelle

On analyse les patterns horaires, journaliers et les acteurs les plus actifs.

In [ ]:
# Distribution des heures d'opération (0h-23h)
heure_dist = df['heure_int'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(13, 5))
colors_h = ['#C63B3B' if (h >= 22 or h < 6) else '#14325A' for h in heure_dist.index]
ax.bar(heure_dist.index, heure_dist.values, color=colors_h, edgecolor='white', width=0.8)
ax.set_title("Distribution des heures d'opération — Rouge = heures nocturnes (22h–6h)")
ax.set_xlabel('Heure de la journée')
ax.set_ylabel("Nombre d'opérations")
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}h' for h in range(24)], fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '04_distribution_heures.png'), dpi=150)
plt.show()

nocturnes = (df['heure_int'] >= 22) | (df['heure_int'] < 6)
print(f"Opérations nocturnes : {nocturnes.sum():,} ({nocturnes.mean()*100:.1f}%)")
print(f"Dont par type :")
print(df[nocturnes]['type_mouvement'].value_counts())

In [ ]:
# Comparaison volumes par jour de la semaine
jours_labels = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
vol_jour = df.groupby('jour_semaine')['quantite_abs'].sum()
nb_jour = df.groupby('jour_semaine').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_j = ['#5B6B7C' if i < 5 else '#C9841A' for i in range(7)]

axes[0].bar(jours_labels, vol_jour.values, color=colors_j, edgecolor='white')
axes[0].set_title('Volume total par jour de la semaine')
axes[0].set_ylabel('Volume total')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(jours_labels, nb_jour.values, color=colors_j, edgecolor='white')
axes[1].set_title("Nombre d'opérations par jour de la semaine")
axes[1].set_ylabel("Nombre d'opérations")
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '05_volumes_par_jour.png'), dpi=150)
plt.show()

In [ ]:
# Top 10 camions les plus actifs
top_camions = df[df['camion_id'].notna()]['camion_id'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_camions.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 10 camions les plus actifs (nombre de mouvements)')
ax.set_xlabel('Nombre de mouvements')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '06_top_camions.png'), dpi=150)
plt.show()

In [ ]:
# Top 10 opérateurs les plus actifs
top_operateurs = df['operateur_id'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_operateurs.plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('Top 10 opérateurs les plus actifs (nombre de mouvements)')
ax.set_xlabel('Nombre de mouvements')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '07_top_operateurs.png'), dpi=150)
plt.show()

# Vérification : opérateurs avec volumes anormalement élevés
vol_operateur = df.groupby('operateur_id')['valeur_mouvement'].sum().sort_values(ascending=False)
mean_op = vol_operateur.mean()
std_op = vol_operateur.std()
suspects = vol_operateur[vol_operateur > mean_op + 2*std_op]
print(f"Opérateurs avec volume > moyenne + 2σ : {len(suspects)}")
if len(suspects) > 0:
    print(suspects.head())

**📝 Observations :**

> La distribution horaire révèle que les opérations se concentrent en journée, avec des pics entre 8h et 18h — cohérent avec des horaires de travail normaux. Les **opérations nocturnes** (22h–6h) méritent attention : elles peuvent être légitimes (livraisons de nuit) mais constituent aussi un signal d'alerte potentiel pour la détection d'anomalies.
>
> La comparaison par jour de la semaine montre que les weekends (samedi/dimanche, en ambre) ont une activité réduite — cohérent avec la fermeture ou la réduction des activités logistiques. Cette information confirme la pertinence de la variable `is_weekend` dans les modèles ML.
>
> Les Top 10 camions et opérateurs montrent une distribution relativement homogène — aucun acteur ne concentre un volume anormalement élevé. L'analyse des volumes par opérateur ne révèle pas de comportements suspects (aucun opérateur au-delà de moyenne + 2σ).
>
> **Décision pour le modèle de détection d'anomalies :** les variables `heure_int`, `is_weekend`, `type_mouvement` et `operateur_id` seront des features importantes pour détecter les mouvements atypiques.

## 5. Analyse par Dépôt et Produit

On analyse les volumes et valeurs financières par dépôt et par produit.

In [ ]:
# Volume total de produits déplacés par dépôt
vol_depot = df.groupby('depot_nom')['quantite_abs'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
vol_depot.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Volume total de produits déplacés par dépôt')
ax.set_xlabel('Volume total (valeur absolue)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '08_volume_par_depot.png'), dpi=150)
plt.show()

In [ ]:
# Valeur financière des mouvements par produit
valeur_produit = df.groupby('produit_nom')['valeur_mouvement'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
valeur_produit.plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('Valeur financière totale des mouvements par produit (USD)')
ax.set_xlabel('Valeur totale (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e9:.1f}B'))
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '09_valeur_par_produit.png'), dpi=150)
plt.show()

In [ ]:
# Heatmap : nombre de mouvements par type et par dépôt
heatmap_data = df.pivot_table(
    values='mouvement_id',
    index='type_mouvement',
    columns='depot_nom',
    aggfunc='count'
).fillna(0)

# Raccourcir les noms de dépôts
heatmap_data.columns = [c.replace('Dépôt ', '').replace('Terminal Portuaire ', 'Terminal ')
                         for c in heatmap_data.columns]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heatmap_data, annot=True, fmt='.0f', cmap='Blues', ax=ax)
ax.set_title('Nombre de mouvements par type et par dépôt')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, '10_heatmap_types_depots.png'), dpi=150)
plt.show()

**📝 Observations :**

> Les volumes par dépôt sont relativement homogènes (entre 10,7M et 13,9M d'unités) — le réseau de dépôts est bien équilibré. Le **Dépôt Atakpamé Centre** a le volume le plus élevé, contrairement au **Dépôt Sokodé** qui est le moins actif.
>
> Par produit, le **Bitume** domine largement en valeur financière (9,56 milliards USD) suivi du **Pétrole Brut** (2,17B) et du **Naphta** (1,79B) — ces 3 produits représentent l'essentiel de la valeur transactionnelle. En revanche les Lubrifiants et l'Huile de Base sont minoritaires en valeur malgré leur importance opérationnelle.
>
> La heatmap confirme que les Sorties Vente et Entrées Livraison sont uniformément réparties entre les dépôts — pas de concentration anormale sur un dépôt spécifique.

## 6. Conclusions et Décisions

---

### 🔍 Observations clés

> 1. **42 529 mouvements** propres sur 10 ans — les valeurs manquantes sur `camion_id` et `bon_commande_ref` sont légitimes et bien comprises.
> 2. Les **Sorties Vente** dominent en nombre (39,5%) et en valeur financière.
> 3. Les **Ajustements Inventaire** négatifs (-33M unités) sont importants — à surveiller comme signal d'anomalie potentielle.
> 4. Les opérations **nocturnes** représentent une part non négligeable — à intégrer comme signal dans le modèle.
> 5. Les weekends sont **moins actifs** — la variable `is_weekend` est pertinente.
> 6. Aucun opérateur ni camion ne présente de comportement anormalement élevé sur les 10 ans.
> 7. Les volumes par dépôt sont **équilibrés** — pas de concentration géographique anormale.
> 8. Le **Bitume** domine la valeur financière — produit stratégique à surveiller en priorité.

---

### ✅ Décisions pour le modèle de détection d'anomalies (Notebook 07)

> - **Variables d'entrée pertinentes** issues de cette table : `heure_int`, `jour_semaine`, `is_weekend`, `type_mouvement` (encodé), `quantite_abs`, `valeur_mouvement`
> - **Signal nocturne** : une Sortie Vente entre 22h et 6h est un comportement atypique à signaler
> - **Ajustements anormaux** : un Ajustement Inventaire avec une quantité absolue > 3σ de la moyenne doit déclencher une alerte
> - **Focus produit** : le Bitume et le Pétrole Brut méritent un seuil d'alerte plus sensible (valeur financière élevée)

---

### 📁 Figures produites

| Fichier | Description |
|---|---|
| `01_repartition_types_mouvements.png` | Camembert et barplot des types |
| `02_valeur_par_type.png` | Valeur financière par type |
| `03_evolution_entrees_sorties.png` | Évolution mensuelle entrées/sorties |
| `04_distribution_heures.png` | Histogramme des heures d'opération |
| `05_volumes_par_jour.png` | Volumes par jour de la semaine |
| `06_top_camions.png` | Top 10 camions actifs |
| `07_top_operateurs.png` | Top 10 opérateurs actifs |
| `08_volume_par_depot.png` | Volume total par dépôt |
| `09_valeur_par_produit.png` | Valeur financière par produit |
| `10_heatmap_types_depots.png` | Heatmap types × dépôts |